In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname = "UNK",
    dt_ns          = 2.0,
    output_dir     = Path("./hbond_results"),
    figures_dir    = Path("./figures"),
)

# PSE file from your trajectory
PSE_PATH = Path("../run01/traj.pse")

# Binding pocket residue IDs (for pocket RMSD)
POCKET_RESIDS = [100, 150, 200, 250]

# H-bond atom selection pairs: (sel1, sel2) in PyMOL syntax
HBOND_PAIRS = [
    ("resi 150 and name OD1", "resname UNK and name N1"),
]

# Column labels for the distance columns (optional)
PAIR_LABELS = ["dist_hbond_1"]

# HBond events CSV from NB02
HBOND_EVENTS_CSV = cfg.output_dir / "hbond_all_events_run01.csv"

SAMPLE_NAME = "run01"
OUT_CSV = cfg.output_dir / f"conformation_{SAMPLE_NAME}.csv"
# ============================================================

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mdatools.pymol_bridge.extractor import PyMOLExtractor

cfg.make_dirs()

extractor = PyMOLExtractor(cfg)

# Extract per-frame RMSD + distances via PyMOL
if not OUT_CSV.exists():
    df_conf = extractor.extract_validation_metrics(
        pse_path=PSE_PATH,
        pocket_resids=POCKET_RESIDS,
        hbond_pairs=HBOND_PAIRS,
        output_csv=OUT_CSV,
        pair_labels=PAIR_LABELS,
    )
else:
    df_conf = pd.read_csv(OUT_CSV)
    df_conf["time_ns"] = df_conf["frame"] * cfg.dt_ns

print(df_conf.head())

In [ ]:
from mdatools.plotting.validation_plots import plot_hbond_timeline

# Load H-bond events
df_hb = pd.read_csv(HBOND_EVENTS_CSV) if HBOND_EVENTS_CSV.exists() else pd.DataFrame()

hbond_frames = set(df_hb["frame"].tolist()) if not df_hb.empty else set()
df_conf["has_hbond"] = df_conf["frame"].isin(hbond_frames)

fig = plot_hbond_timeline(
    df_conf,
    df_hb,
    hbond_label=PAIR_LABELS[0] if PAIR_LABELS else "H-bond",
    dist_col=PAIR_LABELS[0] if PAIR_LABELS else "distance",
    save_path=cfg.figures_dir / f"hbond_timeline_{SAMPLE_NAME}.png",
)
plt.show()

In [ ]:
# Contact fingerprint
import matplotlib.pyplot as plt
from mdatools.plotting.validation_plots import plot_contact_fingerprint

fp_csv = cfg.output_dir / f"contact_fp_{SAMPLE_NAME}.csv"
if fp_csv.exists():
    fp_df = pd.read_csv(fp_csv)
    plot_contact_fingerprint(
        fp_df, SAMPLE_NAME,
        save_path=cfg.figures_dir / f"contact_fp_{SAMPLE_NAME}.png"
    )
    plt.show()
else:
    print(f"Contact fingerprint CSV not found: {fp_csv}")
    print("Run notebook 03_template_selection.ipynb first.")